<img src="https://livetennisapi.com/static/logo-mark.png" alt="Live Tennis API" width="48" height="48">

# Tennis findings, re-tested, on the public sample

Re-run four published tennis findings on the public sample of our point-by-point dataset, with the code behind the page.

[Live Tennis API](https://livetennisapi.com) · [The re-test page](https://blog.livetennisapi.com/tennis-findings-retested) · [retest.json](https://blog.livetennisapi.com/tennis-findings-retested/retest.json) · [The analysis code](https://blog.livetennisapi.com/tennis-findings-retested/retest_findings.py) · [Dataset for research](https://livetennisapi.com/data/academic) · [API docs](https://livetennisapi.com/docs)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/livetennisapi/livetennisapi-data/blob/main/notebooks/findings_retest.ipynb) [![Launch on Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/livetennisapi/livetennisapi-data/HEAD?labpath=notebooks%2Ffindings_retest.ipynb)

This notebook runs the four tests of [Tennis findings, re-tested](https://blog.livetennisapi.com/tennis-findings-retested) on the public sample. First it downloads the sample from Zenodo through its concept DOI and checks the two files. It then imports `retest_findings.py`, the code behind every figure on the page, and calls its functions. It copies none of that code.

Each test has its own section. The section quotes the claim with its citation, runs the test against its baseline and prints the tables. Then it sets the sample's result beside the full-record figures in `retest.json`.

The sample covers June 2026 only, so its intervals are wider than the full record's. Use it to check the code and the published sample figures. The verdicts on the page rest on the full record, which is not public. It is free for non-commercial research on application, from the [dataset page](https://livetennisapi.com/data/academic).

Run the cells in order, top to bottom. You need Python 3.10 or newer and nothing to install. The run takes a few minutes, most of them in the shuffles.

The code is MIT licensed and the published figures are CC BY 4.0. The sample's own licence is on its Zenodo record.

## Get the code and the published figures

The notebook uses `retest_findings.py` and `retest.json` from its own folder. In Colab or Binder the folder has neither, so the first cell downloads both from the page. The second cell checks that the code is the version that produced the published figures.

In [ ]:
import hashlib
import json
import random
import re
import sys
import urllib.request
from collections import Counter
from pathlib import Path

if sys.version_info < (3, 10):  # noqa: UP036 (the notebook also runs outside this repo)
    raise RuntimeError("retest_findings.py needs Python 3.10 or newer")

PAGE = "https://blog.livetennisapi.com/tennis-findings-retested"
HERE = Path.cwd()
DATA = HERE / "data"
# The page's host turns away urllib's default client name, so every request names this notebook.
AGENT = {"User-Agent": "findings-retest-notebook/1.0 (+https://blog.livetennisapi.com/tennis-findings-retested)"}


def get(url: str, timeout: int = 60):
    return urllib.request.urlopen(urllib.request.Request(url, headers=AGENT), timeout=timeout)


def fetch(url: str, path: Path) -> Path:
    """Download url to path, unless an earlier run already did."""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        part = path.with_name(path.name + ".part")
        with get(url, timeout=120) as resp, open(part, "wb") as out:
            while chunk := resp.read(1 << 20):
                out.write(chunk)
        part.replace(path)
    return path


for name in ("retest_findings.py", "retest.json"):
    fetch(f"{PAGE}/{name}", HERE / name)
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

In [ ]:
import retest_findings as rf

published = json.loads((HERE / "retest.json").read_text(encoding="utf-8"))
SAMPLE = published["results"]["public_sample"]
FULL = published["results"]["full"]
VERDICT = {s["id"]: s["verdict"] for s in published["studies"]}
if rf.VERSION != SAMPLE["script_version"]:
    raise RuntimeError(f"retest_findings.py is version {rf.VERSION}, but the published figures come from "
                       f"version {SAMPLE['script_version']}. Download both files from the page again.")
print(f"retest_findings.py version {rf.VERSION}, read from {rf.__file__}")
print(f"Published run: {SAMPLE['label']}, seed {SAMPLE['seed']}, {SAMPLE['point_null_reps']} point "
      f"shuffles and {SAMPLE['game_null_reps']} game shuffles per match")

## Download the public sample

The concept DOI [10.5281/zenodo.22048730](https://doi.org/10.5281/zenodo.22048730) always points at the newest version of the sample. On 19 November 2026 that was version 1.0.3. Its points and matches files are the same files as in version 1.0.2, which the page's commands fetch. The cell checks both against their MD5 checksums. If a later version changes them, the cell falls back to version 1.0.3.

In [ ]:
CONCEPT_DOI = "10.5281/zenodo.22048730"
# The two files behind the published sample figures, with the MD5 checksums Zenodo lists for them.
FILES = {"points_sample_2026-06.csv.gz": "cd484a2d728e3f8c3eb338576e9707bf",
         "matches.csv.gz": "0bf47dc8cd7d359621fa2e117ab932f7"}
VERSION_1_0_3 = "23718893"   # the Zenodo record of version 1.0.3


def zenodo_record(record_id: str) -> dict:
    with get(f"https://zenodo.org/api/records/{record_id}") as resp:
        return json.load(resp)


def md5(path: Path) -> str:
    h = hashlib.md5(usedforsecurity=False)
    with open(path, "rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()


with get(f"https://doi.org/{CONCEPT_DOI}") as resp:
    newest = re.search(r"/records/(\d+)", resp.geturl()).group(1)
record = zenodo_record(newest)
print(f"doi:{CONCEPT_DOI} resolves to version {record['metadata']['version']}, doi:{record['doi']}")
listed = {f["key"]: f for f in record["files"]}
if any(listed.get(k, {}).get("checksum") != "md5:" + v for k, v in FILES.items()):
    print("That version has other files, so this run uses version 1.0.3, the one behind the figures.")
    record = zenodo_record(VERSION_1_0_3)
    listed = {f["key"]: f for f in record["files"]}
for name, want in FILES.items():
    path = fetch(listed[name]["links"]["self"], DATA / name)
    if md5(path) != want:
        path.unlink()
        raise RuntimeError(f"{name} does not match its checksum. Run this cell again to download it again.")
    print(f"{name}: {path.stat().st_size:,} bytes, checksum matches")

## Read the matches

`rf.read_split` joins the two files and `rf.load_matches` rebuilds each match from its score states. The counts should equal the published ones.

In [ ]:
counts = Counter()
source = rf.read_split(str(DATA / "points_sample_2026-06.csv.gz"), str(DATA / "matches.csv.gz"), counts)
matches = rf.load_matches(source, counts)
corpus = rf.corpus_summary(matches, counts)
for key, what in (("matches_read", "Matches in the file"),
                  ("matches_not_eligible", "Left out: not a completed best-of-three singles match"),
                  ("matches_used", "Matches used"),
                  ("matches_with_server_order", "Matches with a known server order")):
    print(f"{what}: {corpus[key]:,} (published: {SAMPLE['corpus'][key]:,})")

Run the sections in order. The shuffles share one seeded stream of random numbers, drawn in the order `rf.analyse` uses.

The helpers below only print tables. Every figure in them comes from `retest_findings.py` or `retest.json`.

Figures are in percentage points unless they end in %. A row marked thin has a 95% interval wider than 4 points or fewer than 500 units behind it, as on the page.

In [ ]:
SEED = SAMPLE["seed"]
POINT_REPS = SAMPLE["point_null_reps"]
GAME_REPS = SAMPLE["game_null_reps"]
rnd = random.Random(SEED).random  # noqa: S311 (a seeded simulation, as in rf.analyse)

GROUPS = {"all": "All six tours", "tour:ATP": "ATP", "tour:WTA": "WTA",
          "tour:Challenger M": "Challenger men", "tour:Challenger W": "Challenger women",
          "tour:ITF M": "ITF men", "tour:ITF W": "ITF women",
          "surface:hard": "Hard courts", "surface:clay": "Clay courts", "surface:grass": "Grass courts"}


def signed(v) -> str:
    """A figure in points, signed the way the page shows it."""
    if v is None:
        return "n/a"
    return "0.00" if round(v, 2) == 0 else f"{v:+.2f}"


def pct(v) -> str:
    return "n/a" if v is None else f"{v:.1f}%"


def span(ci, show=signed) -> str:
    return f"{show(ci[0])} to {show(ci[1])}" if ci else "n/a"


def thin(r) -> str:
    return " (thin)" if r.get("thin") else ""


def share(r) -> str:
    return f"{pct(r['value_pct'])} of {r['units']:,}{thin(r)}" if r else "n/a"


def rows_of(block) -> list:
    return [(g, block[g]) for g in GROUPS if g in block]


def table(header, rows) -> None:
    rows = [tuple(str(c) for c in row) for row in rows]
    widths = [max(len(c) for c in col) for col in zip(header, *rows, strict=True)]
    for i, row in enumerate([header, *rows]):
        cols = zip(row, widths, strict=True)
        print("  ".join(c.ljust(w) if j == 0 else c.rjust(w) for j, (c, w) in enumerate(cols)))
        if i == 0:
            print("  ".join("-" * w for w in widths))


def effect_table(block, unit) -> None:
    """Each group's test against its baseline: the real gap, the shuffled gap and the difference."""
    table(("Group", unit, "Real", "Shuffled", "Beyond chance", "95% interval"),
          [(GROUPS[g], f"{r['units']:,}", signed(r["observed_pp"]), signed(r["null_pp"]),
            signed(r["excess_pp"]) + thin(r), span(r["ci95_pp"])) for g, r in rows_of(block)])


def two_rates(block, a, b) -> None:
    """Two rates measured on the same matches, and their difference."""
    table(("Group", a, b, "Difference", "95% interval"),
          [(GROUPS[g], f"{pct(r['a_pct'])} of {r['a_units']:,}", f"{pct(r['b_pct'])} of {r['b_units']:,}",
            signed(r["difference_pp"]) + thin(r), span(r["ci95_pp"])) for g, r in rows_of(block)])


def where(ci, at) -> str:
    lo, hi = ci
    return "above" if lo > at else "below" if hi < at else "includes"


def compare(sample, full, value, ci, at=0.0, show=signed) -> None:
    """The sample's all-tours figure beside the full record's, and how the two read."""
    mark = "50%" if at else "zero"
    a, b = where(sample[ci], at), where(full[ci], at)
    table(("All six tours", "Figure", "95% interval", "The interval"),
          [("Public sample, June 2026", show(sample[value]), span(sample[ci], show), f"{a} {mark}"),
           ("Full record", show(full[value]), span(full[ci], show), f"{b} {mark}")])
    if a == b:
        verdict = "The two read the same."
    elif a == "includes":
        verdict = "The sample is too small to show what the full record shows."
    elif b == "includes":
        verdict = "The sample shows an effect that the full record does not."
    else:
        verdict = "The two point in opposite directions."
    overlap = sample[ci][0] <= full[ci][1] and full[ci][0] <= sample[ci][1]
    verdict += " Their intervals overlap." if overlap else " Their intervals do not overlap."
    if sample[ci][1] - sample[ci][0] > full[ci][1] - full[ci][0]:
        verdict += " The sample's is wider, since it holds one month of matches."
    print(verdict)


def reproduced(part, key) -> None:
    same = json.loads(json.dumps(part)) == SAMPLE[key]
    print(f"Same figures as results.public_sample.{key} in retest.json: {'yes' if same else 'NO'}")

## 1. Are points independent and identically distributed?

> Points are neither independent nor identically distributed. Winning the previous point has a positive effect on winning the current point. At important points it is more difficult for the server to win the point. The deviations are small, so the iid model is still a good approximation in many cases.

Franc J. G. M. Klaassen and Jan R. Magnus (2001). *Are Points in Tennis Independent and Identically Distributed? Evidence From a Dynamic Binary Panel Data Model*. Journal of the American Statistical Association 96(454), 500-509. [doi:10.1198/016214501753168217](https://doi.org/10.1198/016214501753168217). Data: Wimbledon 1992-1995, almost 90,000 points.

**The test.** Inside each match, the server's chance after a won point is set against after a lost one. Break points are set against other points. Points from deuce on are left out.

**The baseline** is the same games with their points shuffled by `rf.reshuffle_game`. A shuffle keeps each game's score and length and breaks any link between its points. The effect beyond chance is the real gap minus the shuffled one. `rf.study_points` runs both.

In [ ]:
points = rf.study_points(matches, POINT_REPS, rnd)

print("Server's chance after a won point, minus after a lost point")
effect_table(points["prev1"], "Points")
print()
compare(points["prev1"]["all"], FULL["points"]["prev1"]["all"], "excess_pp", "ci95_pp")

print("\n\nServer's chance on break points, minus on other points")
effect_table(points["breakpoint"], "Points")
print()
compare(points["breakpoint"]["all"], FULL["points"]["breakpoint"]["all"], "excess_pp", "ci95_pp")

print(f"\n\nVerdict on the page, from the full record: {VERDICT['points-iid']}")
reproduced(points, "points")

## 2. Is there an advantage in serving first in a set?

> In the men's singles, the first server won 54.7% of first sets. The authors named an advantage in serving first in the first set as the one result that held up. The women's data showed no clear edge. In later sets the first server won less than half the time. The authors put that down to who serves first. The set winner usually closes the set on their own serve, so the weaker player tends to serve first in the next set.

Jan R. Magnus and Franc J. G. M. Klaassen (1999). *On the Advantage of Serving First in a Tennis Set: Four Years at Wimbledon*. Journal of the Royal Statistical Society: Series D (The Statistician) 48(2), 247-256. [doi:10.1111/1467-9884.00186](https://doi.org/10.1111/1467-9884.00186). Data: Wimbledon 1992-1995, almost 90,000 points.

Jan R. Magnus and Franc J. G. M. Klaassen (1996). *Testing some common tennis hypotheses: Four years at Wimbledon*. CentER Discussion Paper 1996-73, Tilburg University (hypothesis 2, the figures quoted here). [full text (PDF, University of Amsterdam)](https://pure.uva.nl/ws/files/2081783/126707_HYPCDP.pdf). Data: 481 Wimbledon singles matches, 1992-1995.

**The test.** For every set with a known first server and winner, how often the first server wins it. For sets 2 and 3, the winner of the previous set, serving first against receiving first.

**The baseline.** This test compares rates, so it needs no shuffle. Its yardstick is an even chance, 50%. For sets 2 and 3 it is the gap between serving first and receiving first. `rf.study_serve_first` runs it.

In [ ]:
serve = rf.study_serve_first(matches)
sets = serve["first_server_wins_set"]

print("How often the player who serves first wins the set")
table(("Group", "Set 1", "Set 2", "Set 3"),
      [(GROUPS[g], *(share(sets[n].get(g)) for n in ("1", "2", "3"))) for g, _ in rows_of(sets["1"])])
for n in ("1", "2"):
    print(f"\nSet {n}, against an even chance")
    compare(sets[n]["all"], FULL["serve_first"]["first_server_wins_set"][n]["all"],
            "value_pct", "ci95_pct", at=50.0, show=pct)

prev = serve["previous_set_winner_wins_serving_vs_receiving"]
print("\n\nSets 2 and 3: the winner of the previous set, serving first against receiving first")
two_rates(prev, "Serving first", "Receiving first")
print()
compare(prev["all"], FULL["serve_first"]["previous_set_winner_wins_serving_vs_receiving"]["all"],
        "difference_pp", "ci95_pp")

print(f"\n\nVerdict on the page, from the full record: {VERDICT['serve-first']}")
reproduced(serve, "serve_first")

## 3. After breaking serve, are you more likely to be broken back?

> Hypothesis 12 of seventeen commonly heard statements: after breaking your opponent's service there is an increased chance that you will lose your own service. The authors found the opposite. After a break the hold rate rose from 81.8% to 83.3% in the men's singles. In the women's singles it rose from 64.3% to 68.3%.

Jan R. Magnus and Franc J. G. M. Klaassen (1996). *Testing some common tennis hypotheses: Four years at Wimbledon*. CentER Discussion Paper 1996-73, Tilburg University. [full text (PDF, University of Amsterdam)](https://pure.uva.nl/ws/files/2081783/126707_HYPCDP.pdf). Data: 481 Wimbledon singles matches, 1992-1995.

**The test.** Every service game straight after the server broke serve in the same set. First the paper's way: its hold rate against all service games. Then inside each match: hold after breaking against hold after the opponent held.

**The baseline** for the second comparison is the same matches with each player's service games shuffled among their own service games in the set, by `rf.shuffled`. A shuffle counts only if every set still ends where it ended. `rf.study_games` runs both.

The page reports that the sample and the full record disagree on the game tests, and that the cause is not known. The verdicts rest on the full record.

In [ ]:
games = rf.study_games(matches, GAME_REPS, rnd)
after = games["hold_after_breaking_vs_all_pooled"]

print("Hold rate in the game after breaking, against all service games (the paper's comparison)")
two_rates(after, "Hold after breaking", "All service games")
print()
compare(after["all"], FULL["games"]["hold_after_breaking_vs_all_pooled"]["all"], "difference_pp", "ci95_pp")

print("\n\nInside each match: hold after breaking, minus hold after the opponent held")
effect_table(games["k1"], "Service games")
print()
compare(games["k1"]["all"], FULL["games"]["k1"]["all"], "excess_pp", "ci95_pp")

print(f"\n\nVerdict on the page, from the full record: {VERDICT['break-back']}")
reproduced(games, "games")

## 4. Do streaks predict the next point or game?

> Momentum is real: recent success raises the chance of winning the next point. The two 2024-2025 papers model the points of the 2023 Wimbledon men's singles and report statistical evidence for momentum. Jackson and Mosurski asked the older form of the question about whole sets: are heavy defeats a sign of psychological momentum or a random effect?

Jingya Wang, Sihang Guo and Yuanyun Zhou (2024). *A multidimensional momentum chain model for tennis matches based on difference equations*. PLOS ONE 19(12), e0316542. [doi:10.1371/journal.pone.0316542](https://doi.org/10.1371/journal.pone.0316542). Data: Wimbledon 2023 men's singles, 7,284 points.

Chang Du, Caiya Zhang and Likai Zhou (2025). *A novel methodological framework for analyzing the momentum effect in tennis singles*. arXiv preprint 2509.01243. [arXiv:2509.01243](https://arxiv.org/abs/2509.01243). Data: Wimbledon 2023 men's singles.

David Jackson and Krzysztof Mosurski (1997). *Heavy Defeats in Tennis: Psychological Momentum or Random Effect?*. CHANCE 10(2), 27-34. [doi:10.1080/09332480.1997.10542019](https://doi.org/10.1080/09332480.1997.10542019). Data: match results by set (see the paper).

**The test.** Two kinds of streak. Points: did the server win the last two or three points of the game? Games: did a player win the last two or three games of the set, and do they then serve or receive? Each streak is set against no streak inside the same match.

**The baselines** are the ones above. The point streaks come from the shuffled points of section 1, the game streaks from the shuffled games of section 3.

In [ ]:
STREAKS = [("points", "prev2", "Server won the last 2 points of the game", "next point"),
           ("points", "prev3", "Server won the last 3 points of the game", "next point"),
           ("games", "serve_k2", "Won the last 2 games of the set", "own service game"),
           ("games", "return_k2", "Won the last 2 games of the set", "return game"),
           ("games", "serve_k3", "Won the last 3 games of the set", "own service game"),
           ("games", "return_k3", "Won the last 3 games of the set", "return game")]
ours = {"points": points, "games": games}

print("All six tours: the streak against no streak, beyond chance")
table(("Streak", "Then", "Public sample", "95% interval", "Full record", "95% interval"),
      [(streak, then, signed(s["excess_pp"]) + thin(s), span(s["ci95_pp"]),
        signed(f["excess_pp"]), span(f["ci95_pp"]))
       for part, key, streak, then in STREAKS
       for s, f in [(ours[part][key]["all"], FULL[part][key]["all"])]])
for part, key, streak, then in STREAKS:
    print(f"\n{streak}, {then}")
    compare(ours[part][key]["all"], FULL[part][key]["all"], "excess_pp", "ci95_pp")

print("\n\nHolding serve after winning the last 3 games of the set, by group")
effect_table(games["serve_k3"], "Service games")

print(f"\n\nVerdict on the page, from the full record: {VERDICT['hot-hand']}")

## Check the whole run

First the table under *Rerun it yourself* on the page, from this run. Then one check of every figure against `results.public_sample` in `retest.json`. The cell stops with an error if any figure differs.

In [ ]:
ROWS = [("Previous point won, effect on the next serve point", "points", "prev1"),
        ("Break point, effect on the server", "points", "breakpoint"),
        ("Hold after breaking, effect beyond chance", "games", "k1"),
        ("Hold after winning 2 games in a row, effect beyond chance", "games", "serve_k2")]


def pts(r) -> str:
    return f"{signed(r['excess_pp'])} pts ({span(r['ci95_pp'])})"


table(("Measure", "Public sample", "Full record"),
      [(label, pts(ours[part][key]["all"]), pts(FULL[part][key]["all"])) for label, part, key in ROWS]
      + [("First server wins set 1", share(sets["1"]["all"]),
          share(FULL["serve_first"]["first_server_wins_set"]["1"]["all"]))])

run = {"script_version": rf.VERSION, "seed": SEED, "point_null_reps": POINT_REPS,
       "game_null_reps": GAME_REPS, "corpus": corpus, "points": points, "serve_first": serve,
       "games": games, "reading": dict(sorted(counts.items()))}
mine = json.loads(json.dumps(rf.public_run(run)))
differ = sorted(k for k in SAMPLE if k != "label" and mine.get(k) != SAMPLE[k])
if differ:
    raise AssertionError(f"These parts differ from results.public_sample in retest.json: {differ}")
print("\nEvery figure of this run matches results.public_sample in retest.json, to the decimal.")